In [11]:
from src.models import LBVSDNN
from src.preprocessing import load_dataset, validate_bundle
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
import joblib
import torch



### THE FINAL DENSE NEURAL NETWORKS

In [4]:
bundle = load_dataset("../data/processed/df.csv")
validate_bundle(bundle)


📊 Preprocessing Validation
Samples        : 3715
Features       : 1036
Descriptors    : 12
Fingerprints   : 1024
Target mean    : 6.7329
Target std     : 1.5795


In [2]:
study_dnn = joblib.load("../data/optim/dnn_optuna_study.pkl")
dnn_best_params = study_dnn.best_params

In [5]:
dnn_hidden_dims = [dnn_best_params[f'h{i}'] for i in range(dnn_best_params['n_layers'])]
dnn_hidden_dims

[128, 448]

In [21]:
final_dnn = LBVSDNN(
    input_dim=bundle.X.shape[1], 
    hidden_dims=dnn_hidden_dims,
    dropout=dnn_best_params['dropout'],
    activation="SiLU"
)
print(final_dnn)

LBVSDNN(
  (feature_extractor): Sequential(
    (0): Linear(in_features=1036, out_features=128, bias=True)
    (1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (2): SiLU()
    (3): Dropout(p=0.10860685748738372, inplace=False)
    (4): Linear(in_features=128, out_features=448, bias=True)
    (5): LayerNorm((448,), eps=1e-05, elementwise_affine=True)
    (6): SiLU()
    (7): Dropout(p=0.05430342874369186, inplace=False)
  )
  (regressor): Linear(in_features=448, out_features=1, bias=True)
)


In [ ]:
optimizer = torch.optim.AdamW(final_dnn.parameters(), lr=dnn_best_params['lr'])
batch_size = dnn_best_params['batch']

### THE Final XGBOOST REGRESSOR 

In [8]:
study_xgb = joblib.load("../data/optim/xgb_optuna_study.pkl")
xgb_best_params = study_xgb.best_params

In [16]:
final_xgb = XGBRegressor(**xgb_best_params, random_state=42)

### THE Final RANDOM FOREST REGRESSOR

In [13]:
study_rf = joblib.load("../data/optim/rf_optuna_study.pkl")
rf_best_params = study_rf.best_params

In [17]:
final_rf = RandomForestRegressor(**rf_best_params, random_state=42)